# LeJEPA Embedding Visualization

Load a trained LeJEPA encoder and visualize the learned embedding space via PCA.

Set `CHECKPOINT_PATH` below to point at your LeJEPA checkpoint.

In [ ]:
#CHECKPOINT_PATH = "../checkpoints/<run_id>/lejepa_last.pth"  # <-- edit this

CHECKPOINT_PATH = "..checkpoints/lejepa_8da619b3/lejepa_epoch_200.pth"


In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

from jepajitfusion.config import DataConfig, EncoderConfig
from jepajitfusion.data.datasets import get_dataset
from jepajitfusion.data.transforms import eval_transform
from jepajitfusion.encoder.vit import VisionTransformer
from jepajitfusion.utils import get_device

## Load checkpoint and build encoder

In [ ]:
device = get_device()
checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)

print(f"Checkpoint keys: {list(checkpoint.keys())}")
print(f"Epoch: {checkpoint['epoch']}")
print(f"Run ID: {checkpoint.get('run_id', 'N/A')}")

In [ ]:
# Build encoder from checkpoint configs (or use defaults for older checkpoints)
enc_cfg = EncoderConfig(**checkpoint["encoder_config"]) if "encoder_config" in checkpoint else EncoderConfig()
ds_cfg = DataConfig(**checkpoint["dataset_config"]) if "dataset_config" in checkpoint else DataConfig()

print(f"Encoder: embed_dim={enc_cfg.embed_dim}, depth={enc_cfg.depth}, heads={enc_cfg.num_heads}")
print(f"Dataset: {ds_cfg.name}, img_size={ds_cfg.img_size}")

encoder = VisionTransformer(
    img_size=ds_cfg.img_size,
    patch_size=enc_cfg.patch_size,
    in_channels=ds_cfg.num_channels,
    embed_dim=enc_cfg.embed_dim,
    depth=enc_cfg.depth,
    num_heads=enc_cfg.num_heads,
    mlp_ratio=enc_cfg.mlp_ratio,
)

# Prefer EMA weights if available
if "ema_state_dicts" in checkpoint and checkpoint["ema_state_dicts"]:
    encoder.load_state_dict(checkpoint["ema_state_dicts"][0])
    print("Loaded EMA encoder weights")
else:
    encoder.load_state_dict(checkpoint["model_state_dict"])
    print("Loaded encoder weights")

encoder = encoder.to(device)
encoder.eval()
print(f"Encoder on {device}")

## Training loss curve

Plot the training loss over epochs if `train_losses` is available in the checkpoint.

In [ ]:
train_losses = checkpoint.get("train_losses")
val_losses = checkpoint.get("val_losses")

if train_losses:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    epochs = range(1, len(train_losses) + 1)
    ax1.plot(epochs, train_losses, linewidth=0.8, label="Train")
    if val_losses:
        # Val losses are recorded every validate_every epochs
        val_interval = max(1, len(train_losses) // len(val_losses)) if val_losses else 1
        val_epochs = [val_interval * (i + 1) for i in range(len(val_losses))]
        ax1.plot(val_epochs, val_losses, linewidth=0.8, label="Val")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Loss")
    ax1.set_title("Training loss")
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # Log-scale view
    ax2.plot(epochs, train_losses, linewidth=0.8, label="Train")
    if val_losses:
        ax2.plot(val_epochs, val_losses, linewidth=0.8, label="Val")
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("Loss (log scale)")
    ax2.set_title("Training loss (log scale)")
    ax2.set_yscale("log")
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    fig.suptitle(
        f"LeJEPA training — {ds_cfg.name} (run {checkpoint.get('run_id', 'N/A')})",
        fontsize=12,
    )
    plt.tight_layout()
    plt.show()

    print(f"Final loss: {train_losses[-1]:.6f} (epoch {len(train_losses)})")
    print(f"Min loss:   {min(train_losses):.6f} (epoch {train_losses.index(min(train_losses)) + 1})")
    if val_losses:
        print(f"Min val loss: {min(val_losses):.6f}")
else:
    print("No train_losses in checkpoint (older checkpoint format). Re-train to get loss curves.")

## Load dataset and encode images

In [ ]:
MAX_IMAGES = 2000  # cap for speed; increase if you want denser plots
BATCH_SIZE = 128

transform = eval_transform(ds_cfg.img_size)
train_ds, test_ds = get_dataset(
    ds_cfg.name, transform=transform, data_dir="../downloads", test_size=ds_cfg.test_size
)

# Use test set for eval
dataset = test_ds
n_images = min(len(dataset), MAX_IMAGES)
print(f"Encoding {n_images} images from {ds_cfg.name} test set...")

In [ ]:
from torch.utils.data import DataLoader, Subset

subset = Subset(dataset, range(n_images))
loader = DataLoader(subset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

all_embeddings = []
all_labels = []

with torch.no_grad():
    for images, labels in loader:
        images = images.to(device)
        emb = encoder(images)  # (B, embed_dim)
        all_embeddings.append(emb.cpu())
        all_labels.append(labels)

embeddings = torch.cat(all_embeddings, dim=0).numpy()  # (N, embed_dim)
labels = torch.cat(all_labels, dim=0).numpy()           # (N,)

print(f"Embeddings shape: {embeddings.shape}")
print(f"Unique labels: {len(np.unique(labels))}")

## PCA projection

In [ ]:
pca = PCA(n_components=2)
proj_2d = pca.fit_transform(embeddings)

print(f"Explained variance: {pca.explained_variance_ratio_[0]:.1%}, {pca.explained_variance_ratio_[1]:.1%}")
print(f"Total explained: {sum(pca.explained_variance_ratio_):.1%}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

unique_labels = np.unique(labels)
if len(unique_labels) <= 20:
    # Color by class if few enough classes
    for label in unique_labels:
        mask = labels == label
        ax.scatter(proj_2d[mask, 0], proj_2d[mask, 1], s=8, alpha=0.6, label=str(label))
    ax.legend(markerscale=3, fontsize=8)
else:
    # Too many classes — just plot all points
    ax.scatter(proj_2d[:, 0], proj_2d[:, 1], s=8, alpha=0.4, c=labels, cmap="tab20")

ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%})")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%})")
ax.set_title(f"LeJEPA embeddings — {ds_cfg.name} (epoch {checkpoint['epoch']})")
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

## Patch-level PCA heatmaps

Project each patch token into the first 3 principal components and map them to RGB.
This shows *what* the encoder has learned to attend to spatially — similar to
the visualizations in DINO / DINOv2.

PCA is fit on patch tokens pooled across many images, then applied per-image
to produce a spatial heatmap at patch resolution, upsampled for display.

In [ ]:
PCA_IMAGES = 500  # images to fit patch-level PCA on
N_PCS = 3         # first 3 PCs → RGB channels

patch_h = ds_cfg.img_size // enc_cfg.patch_size  # patches along height
patch_w = ds_cfg.img_size // enc_cfg.patch_size  # patches along width
print(f"Patch grid: {patch_h}x{patch_w} = {patch_h * patch_w} tokens per image")

# Collect patch tokens (skip CLS at index 0) across images
pca_subset = Subset(dataset, range(min(len(dataset), PCA_IMAGES)))
pca_loader = DataLoader(pca_subset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

all_patch_tokens = []
with torch.no_grad():
    for images, _ in pca_loader:
        images = images.to(device)
        tokens = encoder(images, return_all_tokens=True)  # (B, N+1, D)
        patch_tokens = tokens[:, 1:, :]                   # (B, N, D) — drop CLS
        all_patch_tokens.append(patch_tokens.cpu())

all_patch_tokens = torch.cat(all_patch_tokens, dim=0)  # (n_imgs, N, D)
n_imgs = all_patch_tokens.shape[0]

# Fit PCA on all patch tokens pooled together
flat_patches = all_patch_tokens.reshape(-1, all_patch_tokens.shape[-1]).numpy()  # (n_imgs*N, D)
patch_pca = PCA(n_components=N_PCS)
patch_pca.fit(flat_patches)

print(f"Patch PCA explained variance: {[f'{v:.1%}' for v in patch_pca.explained_variance_ratio_]}")
print(f"Fit on {flat_patches.shape[0]} patch tokens from {n_imgs} images")

In [ ]:
from jepajitfusion.data.transforms import reverse_transform
from PIL import Image

rev_t = reverse_transform()

def patch_pca_heatmap(img_patch_tokens, pca_model, patch_h, patch_w, img_size):
    """Project patch tokens through PCA and return an upsampled RGB image.

    Args:
        img_patch_tokens: (N, D) patch embeddings for one image.
        pca_model: fitted PCA with n_components=3.
        patch_h, patch_w: spatial grid dimensions.
        img_size: target pixel size for upsampling.

    Returns:
        (img_size, img_size, 3) numpy array in [0, 1].
    """
    projected = pca_model.transform(img_patch_tokens)     # (N, 3)
    spatial = projected.reshape(patch_h, patch_w, 3)

    # Normalize each channel to [0, 1] independently
    for c in range(3):
        ch = spatial[:, :, c]
        lo, hi = ch.min(), ch.max()
        spatial[:, :, c] = (ch - lo) / (hi - lo + 1e-8)

    # Upsample from patch grid to pixel resolution via nearest-neighbor PIL
    heatmap_pil = Image.fromarray((spatial * 255).astype(np.uint8), mode="RGB")
    heatmap_pil = heatmap_pil.resize((img_size, img_size), Image.NEAREST)
    return np.array(heatmap_pil).astype(np.float32) / 255.0


# Pick sample images spread across the dataset
N_DISPLAY = 8
display_idxs = np.linspace(0, min(n_imgs, len(pca_subset)) - 1, N_DISPLAY, dtype=int)

fig, axes = plt.subplots(N_DISPLAY, 3, figsize=(9, 3 * N_DISPLAY))
col_titles = ["Input image", "PCA heatmap (PC1=R, PC2=G, PC3=B)", "Overlay"]

for row, idx in enumerate(display_idxs):
    # Original image
    img_tensor, _ = pca_subset[idx]
    orig_pil = rev_t(img_tensor)
    orig_np = np.array(orig_pil).astype(np.float32) / 255.0

    # PCA heatmap
    tokens_np = all_patch_tokens[idx].numpy()  # (N, D)
    heatmap = patch_pca_heatmap(tokens_np, patch_pca, patch_h, patch_w, ds_cfg.img_size)

    # Blended overlay
    overlay = 0.5 * orig_np + 0.5 * heatmap

    for col, img in enumerate([orig_np, heatmap, overlay]):
        axes[row, col].imshow(img)
        axes[row, col].axis("off")
        if row == 0:
            axes[row, col].set_title(col_titles[col], fontsize=10)

fig.suptitle(
    f"Patch-level PCA — {ds_cfg.name} (epoch {checkpoint['epoch']})\n"
    f"Explained variance: {', '.join(f'PC{i+1}={v:.1%}' for i, v in enumerate(patch_pca.explained_variance_ratio_))}",
    fontsize=12,
)
plt.tight_layout()
plt.show()

In [ ]:
# Per-channel breakdown: show each PC as a separate grayscale heatmap
N_DETAIL = 4  # images to show in detail
detail_idxs = np.linspace(0, min(n_imgs, len(pca_subset)) - 1, N_DETAIL, dtype=int)

fig, axes = plt.subplots(N_DETAIL, 5, figsize=(15, 3 * N_DETAIL))
col_titles = ["Input", "PC1", "PC2", "PC3", "RGB combined"]

for row, idx in enumerate(detail_idxs):
    img_tensor, _ = pca_subset[idx]
    orig_pil = rev_t(img_tensor)
    orig_np = np.array(orig_pil).astype(np.float32) / 255.0

    tokens_np = all_patch_tokens[idx].numpy()
    projected = patch_pca.transform(tokens_np)  # (N, 3)

    axes[row, 0].imshow(orig_np)
    axes[row, 0].axis("off")

    # Individual PC channels as grayscale heatmaps
    for pc in range(3):
        spatial = projected[:, pc].reshape(patch_h, patch_w)
        lo, hi = spatial.min(), spatial.max()
        spatial = (spatial - lo) / (hi - lo + 1e-8)

        # Upsample to image resolution
        hm_pil = Image.fromarray((spatial * 255).astype(np.uint8), mode="L")
        hm_pil = hm_pil.resize((ds_cfg.img_size, ds_cfg.img_size), Image.NEAREST)

        axes[row, 1 + pc].imshow(np.array(hm_pil), cmap="inferno")
        axes[row, 1 + pc].axis("off")

    # RGB combined
    heatmap = patch_pca_heatmap(tokens_np, patch_pca, patch_h, patch_w, ds_cfg.img_size)
    axes[row, 4].imshow(heatmap)
    axes[row, 4].axis("off")

    if row == 0:
        for col, title in enumerate(col_titles):
            axes[row, col].set_title(title, fontsize=10)

fig.suptitle("Per-component spatial heatmaps (inferno colormap)", fontsize=12)
plt.tight_layout()
plt.show()

## Explained variance spectrum

A good encoder should use many dimensions — if the first 2 PCs explain almost everything, the representation may have collapsed.

In [ ]:
pca_full = PCA(n_components=min(50, embeddings.shape[1]))
pca_full.fit(embeddings)

cumulative = np.cumsum(pca_full.explained_variance_ratio_)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.bar(range(len(pca_full.explained_variance_ratio_)), pca_full.explained_variance_ratio_)
ax1.set_xlabel("Principal Component")
ax1.set_ylabel("Explained Variance Ratio")
ax1.set_title("Per-component explained variance")

ax2.plot(range(len(cumulative)), cumulative, "o-", markersize=4)
ax2.axhline(y=0.9, color="r", linestyle="--", alpha=0.5, label="90%")
ax2.set_xlabel("Number of Components")
ax2.set_ylabel("Cumulative Explained Variance")
ax2.set_title("Cumulative explained variance")
ax2.legend()

plt.tight_layout()
plt.show()

n_90 = np.searchsorted(cumulative, 0.9) + 1
print(f"Components for 90% variance: {n_90} / {embeddings.shape[1]}")

## Nearest neighbors in embedding space

Pick a few query images and show their nearest neighbors — a quick sanity check that similar images are nearby.

In [ ]:
from jepajitfusion.data.transforms import reverse_transform

N_QUERIES = 5
N_NEIGHBORS = 5

rev_t = reverse_transform()

# Cosine similarity matrix
emb_norm = embeddings / (np.linalg.norm(embeddings, axis=1, keepdims=True) + 1e-8)
sim = emb_norm @ emb_norm.T  # (N, N)

# Pick evenly-spaced queries
query_idxs = np.linspace(0, n_images - 1, N_QUERIES, dtype=int)

fig, axes = plt.subplots(N_QUERIES, N_NEIGHBORS + 1, figsize=(2 * (N_NEIGHBORS + 1), 2 * N_QUERIES))

for row, qi in enumerate(query_idxs):
    # Query image
    img_tensor, _ = subset[qi]
    axes[row, 0].imshow(rev_t(img_tensor))
    axes[row, 0].set_title("Query", fontsize=9)
    axes[row, 0].axis("off")

    # Top-k neighbors (skip self)
    neighbors = np.argsort(-sim[qi])  # descending similarity
    neighbor_idx = 0
    col = 1
    for ni in neighbors:
        if ni == qi:
            continue
        img_tensor, _ = subset[int(ni)]
        axes[row, col].imshow(rev_t(img_tensor))
        axes[row, col].set_title(f"{sim[qi, ni]:.2f}", fontsize=8)
        axes[row, col].axis("off")
        col += 1
        if col > N_NEIGHBORS:
            break

fig.suptitle("Nearest neighbors in LeJEPA embedding space (cosine similarity)", fontsize=12)
plt.tight_layout()
plt.show()